<a href="https://colab.research.google.com/github/RvXp/Topicos-Especiais-em-IA-LLM/blob/main/LN_Latex_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets torch faiss-cpu sentence-transformers

In [ ]:
import torch
import gc

# Definição global do dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu" # Identifica se há uma GPU disponível.

def preparar_ambiente(): # Liberar memória da GPU
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Rodando em: {device}")

preparar_ambiente()

Rodando em: cuda


In [ ]:
import torch
from datasets import Dataset
# Criar um dataset em formato de dicionário (in para pergunta, out para fórmula).
novos_exemplos = [
    # CAPÍTULO 1: MATRIZES
    {"in": "Defina matriz quadrada de ordem n.", "out": "A = [a_{ij}]_{n \\times n} \\text{ onde } m = n"},
    {"in": "O que caracteriza uma matriz identidade?", "out": "I_n = [a_{ij}] \\text{ onde } a_{ii}=1 \\text{ e } a_{ij}=0 \\text{ para } i \\neq j"},
    {"in": "Defina matriz triangular superior.", "out": "a_{ij}=0 \\text{ para } i > j"},
    {"in": "Quando uma matriz é considerada simétrica?", "out": "A = A^T \\text{ ou } a_{ij} = a_{ji}"},
    {"in": "Defina a soma de duas matrizes A e B.", "out": "A + B = [a_{ij} + b_{ij}]"},
    {"in": "Qual a propriedade da transposta de um produto de matrizes?", "out": "(AB)^T = B^T A^T"},

    # CAPÍTULO 2: SISTEMAS DE EQUAÇÕES LINEARES
    {"in": "O que são sistemas lineares equivalentes?", "out": "S_1 \\sim S_2 \\iff \\text{possuem o mesmo conjunto solução}"},
    {"in": "Defina o posto de uma matriz A.", "out": "p = \\text{número de linhas não nulas da matriz na forma escada}"},
    {"in": "Qual a condição para um sistema linear ser incompatível?", "out": "\\text{posto}(A) < \\text{posto}(A|B)"},
    {"in": "Quando um sistema linear admite uma única solução?", "out": "\\text{posto}(A) = \\text{posto}(A|B) = n"},
    {"in": "Defina a nulidade de uma matriz A.", "out": "\\nu = n - p"},

    # CAPÍTULO 3: DETERMINANTE E MATRIZ INVERSA
    {"in": "Defina o determinante de uma matriz 2x2.", "out": "\\det A = a_{11}a_{22} - a_{12}a_{21}"},
    {"in": "Qual o determinante da matriz identidade?", "out": "\\det I_n = 1"},
    {"in": "Escreva o desenvolvimento de Laplace para a i-ésima linha.", "out": "\\det A = \\sum_{j=1}^n a_{ij} \\Delta_{ij}"},
    {"in": "O que é o cofator de um elemento a_{ij}?", "out": "\\Delta_{ij} = (-1)^{i+j} \\det A_{ij}"},
    {"in": "Qual a relação entre o determinante de A e de sua transposta?", "out": "\\det A = \\det A^T"},
    {"in": "Qual a condição necessária e suficiente para uma matriz ser inversível?", "out": "\\det A \\neq 0"},
    {"in": "Qual a relação entre o determinante do produto e os determinantes individuais?", "out": "\\det(AB) = \\det A \\cdot \\det B"},

    # CAPÍTULO 4: ESPAÇO VETORIAL
    {"in": "Defina um subespaço vetorial W de V.", "out": "\\forall u, v \\in W, \\forall \\alpha, \\beta \\in \\mathbb{R}, \\alpha u + \\beta v \\in W"},
    {"in": "O que constitui uma base de um espaço vetorial?", "out": "\\text{Conjunto LI que gera o espaço V}"},
    {"in": "Defina dependência linear (LD) de um conjunto de vetores.", "out": "\\sum \\alpha_i v_i = 0 \\text{ com algum } \\alpha_i \\neq 0"},
    {"in": "O que é a dimensão de um espaço vetorial?", "out": "\\dim V = \\text{número de vetores de qualquer base de V}"},

    # CAPÍTULO 5 e 9: TRANSFORMAÇÕES LINEARES E OPERADORES
    {"in": "Defina uma transformação linear T: V -> W.", "out": "T(u+v) = T(u)+T(v) \\text{ e } T(\\alpha u) = \\alpha T(u)"},
    {"in": "O que é a imagem de uma transformação linear?", "out": "Im(T) = \\{w \\in W \\mid T(v) = w, v \\in V\\}"},
    {"in": "Defina um operador ortogonal.", "out": "\\langle T(u), T(v) \\rangle = \\langle u, v \\rangle"},
    {"in": "Qual a condição para um operador ser auto-adjunto?", "out": "T = T^*"},

    # CAPÍTULO 6 e 7: AUTOVALORES E DIAGONALIZAÇÃO
    {"in": "Defina autovetor e autovalor de um operador T.", "out": "T(v) = \\lambda v, v \\neq 0"},
    {"in": "Como encontrar os autovalores de uma matriz?", "out": "\\det(A - \\lambda I) = 0"},
    {"in": "O que é uma matriz diagonalizável?", "out": "A \\sim D \\iff \\exists P \\mid D = P^{-1}AP"},
    {"in": "Defina o polinômio minimal de uma matriz.", "out": "m(\\lambda) \\text{ é o polinômio de menor grau tal que } m(A) = 0"},

    # CAPÍTULO 8: PRODUTO INTERNO
    {"in": "Defina o produto interno usual em R^n.", "out": "\\langle u, v \\rangle = \\sum_{i=1}^n u_i v_i"},
    {"in": "Defina vetores ortogonais.", "out": "\\langle u, v \\rangle = 0"},
    {"in": "O que é o complemento ortogonal de um subespaço W?", "out": "W^{\\perp} = \\{v \\in V \\mid \\langle v, w \\rangle = 0, \\forall w \\in W\\}"},
    {"in": "Descreva o processo de ortogonalização de Gram-Schmidt.", "out": "g_n = v_n - \\sum_{i=1}^{n-1} \\frac{\\langle v_n, g_i \\rangle}{\\|g_i\\|^2} g_i"},

    # CAPÍTULO 10: FORMAS QUADRÁTICAS
    {"in": "Escreva a forma bilinear simétrica associada a A.", "out": "f(u, v) = u^T A v"},
    {"in": "Defina uma forma quadrática definida positiva.", "out": "Q(x) > 0, \\forall x \\neq 0"},

    # CAPÍTULO 12: EQUAÇÕES DIFERENCIAIS
    {"in": "Escreva a solução geral de X' = AX para A com autovalores distintos.", "out": "X(t) = \\sum c_i e^{\\lambda_i t} v_i"},
    {"in": "Defina um sistema linear homogêneo de EDOs.", "out": "\\frac{dX}{dt} = AX"},

    # CAPÍTULO 13: PROCESSOS ITERATIVOS
    {"in": "Defina a norma de Frobenius de uma matriz.", "out": "\\|A\\|_F = \\sqrt{\\sum |a_{ij}|^2}"},
    {"in": "Escreva a iteração do método de Gauss-Seidel.", "out": "x^{(k+1)} = (D+L)^{-1}(B - Ux^{(k)})"},
    {"in": "Qual a condição de convergência para métodos iterativos?", "out": "\\rho(M) < 1 \\text{ (raio espectral)}"},

    # CAPÍTULO 14: PROGRAMAÇÃO LINEAR
    {"in": "O que é um conjunto convexo?", "out": "\\forall x, y \\in C, \\lambda x + (1-\\lambda)y \\in C, \\lambda \\in [0,1]"},
    {"in": "Defina um ponto extremo de um conjunto convexo.", "out": "\\text{Ponto que não pode ser combinação linear de outros dois}"},
    {"in": "O que é a região viável em um problema de PL?", "out": "\\text{Intersecção de todos os semi-espaços das restrições}"},
    {"in": "O que caracteriza uma solução básica factível?", "out": "\\text{Solução que corresponde a um vértice da região viável}"},

    # CAPÍTULO 1.5: CADEIAS DE MARKOV
    {"in": "O que é uma matriz estocástica?", "out": "p_{ij} \\ge 0 \\text{ e } \\sum_i p_{ij} = 1"},
    {"in": "Defina uma cadeia de Markov regular.", "out": "\\exists n \\mid T^n \\text{ possui apenas elementos positivos}"},
    {"in": "Como encontrar o estado estacionário de uma cadeia de Markov?", "out": "V = TV"},
    {"in": "O que representa o vetor de probabilidades v^{(n)}?", "out": "v^{(n)} = T^n v^{(1)}"},
]


# Transformando em objeto Dataset do HuggingFace
dataset = Dataset.from_list(novos_exemplos)

In [ ]:
from sentence_transformers import SentenceTransformer # Transformar texto em embeddings
import faiss # Busca vetorial mais rapida
import numpy as np

# Modelo de embedding para transformar perguntas em vetores
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2') # Escolhido porque o conteúdo é em português e contém símbolos matemáticos.

# Criando o índice de busca (FAISS)
questions = [ex["in"] for ex in novos_exemplos]
embeddings = embed_model.encode(questions)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

def retrieve_latex(query, k=1): # Busca o exemplo mais similar no banco para servirvindo de base para a resposta da IA.
    query_vector = embed_model.encode([query])
    D, I = index.search(query_vector, k)
    return novos_exemplos[I[0][0]]["out"]

# Teste do RAG
print(f"RAG Retrieval: {retrieve_latex('Como definir matriz quadrada?')}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG Retrieval: A = [a_{ij}]_{n \times n} \text{ onde } m = n


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer

model_name = "gpt2" # GPT-2 por ser leve, e e bom para tarefas simples como formatação LaTex
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

def tokenize_function(examples): #
    # Formato: <pergunta> -> <equação>
    texts = [f"Pergunta: {i} Resposta: {o}{tokenizer.eos_token}" for i, o in zip(examples["in"], examples["out"])]

    tokenized = tokenizer(texts, padding="max_length", truncation=True, max_length=64) # Controle de tamanho para processamento em Batches

    # Os "rótulos" (labels) para o treino são os próprios tokens de entrada deslocados.
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

# Re-mapeia o dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/49 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./slm_linear_algebra",
    per_device_train_batch_size=4,
    num_train_epochs=10, # É um dataset pequeno, poucas épocas bastam
    logging_steps=10,
    save_steps=50,
    learning_rate=5e-5, # Taxa baixa para não "destruir" o conhecimento prévio do GPT-2
)

trainer = Trainer( # Padrão de treino
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Step,Training Loss
10,4.320681
20,2.301835
30,2.046906
40,1.621802
50,1.450770
60,1.335971
70,1.423777
80,1.157772
90,1.139072
100,1.092935


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=130, training_loss=1.6040694310114934, metrics={'train_runtime': 67.826, 'train_samples_per_second': 7.224, 'train_steps_per_second': 1.917, 'total_flos': 16004136960000.0, 'train_loss': 1.6040694310114934, 'epoch': 10.0})

In [ ]:
def generate_algebra_response(pergunta):
    # 1. Define o dispositivo (GPU se disponível, senão CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # 2. Recupera o conhecimento do RAG (Contexto)
    contexto = retrieve_latex(pergunta)

    # 3. Constrói o Prompt
    # Adicionamos um separador claro para a SLM saber onde começar a gerar
    prompt = f"Pergunta: {pergunta}\nContexto: {contexto}\nResposta Curta:"

    # 4. Tokenização e envio para o device correto
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 5. Geração
    # pad_token_id é importante para evitar avisos em modelos como GPT-2
    output = model.generate(
    **inputs,
    max_new_tokens=20,
    eos_token_id=tokenizer.eos_token_id, # Para quando encontrar o fim da frase
    pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# Teste
pergunta_usuario = "O que é uma matriz simétrica?"
print(generate_algebra_response(pergunta_usuario))

Pergunta: O que é uma matriz simétrica?
Contexto: A = A^T \text{ ou } a_{ij} = a_{ji}
Resposta Curta: A_{ij} = a_{ij}
